# Level 2 - Categorization

Three classes: 0 = real, 1 = generated (fully AI), 2 = local_edit. Metric is macro F1,
per-class one-vs-rest averaged unweighted.

The images are byte-identical to Level 1's across all three splits, including the 993
test images, and Level 2's classes refine Level 1's labels with no inconsistency
(0 -> 0, 1,2 -> 1). So a few things carry straight over:

1. Real vs fake is essentially solved. Level 1's 5-fold ensemble scored 0.99731 on this
   test set, roughly 3-4 errors on 0 vs {1,2}, and called 332 images real against the
   331 that balanced thirds implies. The difficulty here is class 1 vs class 2, which
   Level 1 never had to separate.
2. Classes are exactly balanced, 2619/2619/2619 train and 326/326/326 val, so macro F1
   and accuracy track each other and argmax needs no prior correction. On Level 1 a
   fitted threshold was worth ~0.08 F1.
3. The signal is in the pixels, not the container. A metadata-only model came out at
   AUC 0.692 on Level 1, since the dataset was re-encoded uniformly with ancillary
   chunks stripped. Not worth repeating.
4. No resizing, D4-only augmentation. Cat 2 is a real photo with a small altered region
   and downscaling destroys the resampling evidence. All images are 640x640 RGB PNG.
5. Tiled inference does not work here. On Level 1, 3x3 320px tiles with max/mean/top-2
   pooling came out 2-5x worse than whole-image, 39 errors against 8 for max. Taking
   the max of nine estimates amplifies noise as readily as signal.
6. The OOF noise floor is about +/-2 errors. Re-running an identical Level 1 config
   gave 10 errors against 8, because cudnn.benchmark plus AMP make kernel selection
   nondeterministic. Any A/B needs a repeated-identical-config baseline first, or
   differences under ~4 errors are unreadable.

Runtime: T4 x2, internet on once for the pretrained weights. Not the P100, the torch
2.10 build here has no sm_60 kernels. 5-fold cv at 3 epochs takes about 90 min.

## Config

In [ ]:
class CFG:
    seed = 42
    n_folds = 5
    n_classes = 3

    # Encoder. Has to be a CNN if eval_size != train_size, since ViT and Swin have
    # fixed position embeddings and break on a resolution change.
    model_name = "tf_efficientnetv2_s.in21k_ft_in1k"
    pretrained = True
    drop_rate = 0.3
    drop_path_rate = 0.2

    # 6 channels: RGB + a fixed SRM high-pass residual, which exposes the local
    # resampling and blending artefacts that separate a local edit from a fully
    # generated image. On Level 1 this was worse alone (17 errors against 8-10) but
    # shared only 1 error with plain RGB. Here class 2 is its own class rather than
    # being merged into "fake", and the residual earns its place.
    use_srm = True
    srm_gain = 10.0

    # native; no cropping, no resizing
    train_size = 640
    eval_size = 640

    # Level 1 saturated at epoch 1 of 8, so 3 leaves margin past saturation and keeps
    # 5-fold affordable.
    epochs = 3
    batch_size = 8
    grad_accum = 4         # effective batch = 32
    eval_batch_size = 16
    lr = 2e-4
    weight_decay = 1e-2
    warmup_frac = 0.1
    max_grad_norm = 1.0
    label_smoothing = 0.0
    num_workers = 4
    amp = True

    out_dir = "/kaggle/working"


CLASS_NAMES = ["real", "generated", "local_edit"]
CLASS_LABELS = [0, 1, 2]


## Imports

In [ ]:
import gc
import os
import random
import time

import cv2
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Dataset

# cv2 spawns its own thread pool which fights the DataLoader workers.
cv2.setNumThreads(0)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP_ENABLED = CFG.amp and DEVICE == "cuda"


def autocast():
    return torch.amp.autocast("cuda", dtype=torch.float16, enabled=AMP_ENABLED)


def seed_everything(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(CFG.seed)
# benchmark=True picks the fastest conv kernels. It is also why the OOF error count
# moves by ~2 between identical runs, as noted in the header.
torch.backends.cudnn.benchmark = True

os.makedirs(CFG.out_dir, exist_ok=True)


## Data paths

Column-name trap: ground_truth.csv names the label column `class`, but the submission
wants `classe`.

In [ ]:
DATA_ROOT = "/kaggle/input/gensivana-real-or-fake-level-2-classification"
TEST_DIR = os.path.join(DATA_ROOT, "test", "images")


def load_ground_truth(split: str) -> pd.DataFrame:
    """Load a split's ground_truth.csv, normalising the label column to `label`."""
    df = pd.read_csv(os.path.join(DATA_ROOT, split, "ground_truth.csv"))
    # Provided files use `class`, the submission wants `classe`. Accept either.
    label_col = next((c for c in ("class", "classe", "label") if c in df.columns), None)
    if label_col is None:
        raise KeyError(f"No label column in {split}/ground_truth.csv: {list(df.columns)}")
    df = df.rename(columns={label_col: "label"})
    df["label"] = df["label"].astype(int)
    df["dir"] = os.path.join(DATA_ROOT, split, "images")
    return df[["image_id", "label", "dir"]]


train_df = load_ground_truth("train")
val_df = load_ground_truth("val")

sample_sub = pd.read_csv(os.path.join(DATA_ROOT, "sample_submission.csv"))
test_df = pd.DataFrame({"image_id": sample_sub["image_id"].tolist(), "dir": TEST_DIR})

print(f"train {len(train_df):>5}   {dict(train_df.label.value_counts().sort_index())}")
print(f"val   {len(val_df):>5}   {dict(val_df.label.value_counts().sort_index())}")
print(f"test  {len(test_df):>5}")

# Fail loudly now rather than after an hour of training.
assert train_df.image_id.is_unique and val_df.image_id.is_unique
assert set(train_df.image_id) & set(val_df.image_id) == set(), "train/val leak"


## Baselines

Macro F1 punishes ignoring a class, so the degenerate baselines sit well below Level
1's 0.800. Printed every run so a mediocre score cannot look good.

In [ ]:
def macro_f1_of_constant_prediction(y_true: np.ndarray, cls: int) -> float:
    """Macro F1 obtained by predicting a single class everywhere."""
    return f1_score(y_true, np.full_like(y_true, cls), average="macro",
                    labels=CLASS_LABELS, zero_division=0)


val_labels = val_df.label.values
for c, name in enumerate(CLASS_NAMES):
    print(f"always predict {name:11}: macro F1 "
          f"{macro_f1_of_constant_prediction(val_labels, c):.4f}")
print(f"perfect                  : macro F1 1.0000")
print("\nLevel 1 reference: real-vs-fake on this exact test set scored 0.99731,")
print("so class 0 should be near-perfect and the score hinges on class 1 vs 2.")


## Dataset

No resizing anywhere. Augmentation is D4 only, flips and rot90, which permutes pixels
without resampling them. No colour jitter, blur or JPEG-style augmentation, those would
destroy the signal being detected.

Images come out as float in [0, 1]. Normalisation and the SRM residual happen inside
the model so training and inference cannot drift apart.

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, df: pd.DataFrame, size: int, train: bool):
        self.ids = df.image_id.tolist()
        self.dirs = df.dir.tolist()
        self.labels = df.label.tolist() if "label" in df.columns else None
        self.size = size
        self.train = train

    def __len__(self) -> int:
        return len(self.ids)

    def _read(self, idx: int) -> np.ndarray:
        path = os.path.join(self.dirs[idx], self.ids[idx])
        img = cv2.imread(path, cv2.IMREAD_COLOR)
        if img is None:
            raise RuntimeError(f"failed to read {path}")
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    def __getitem__(self, idx: int):
        img = self._read(idx)
        h, w = img.shape[:2]

        if self.train:
            # Random crop only when the target is smaller than the source.
            if self.size < min(h, w):
                y = random.randint(0, h - self.size)
                x = random.randint(0, w - self.size)
                img = img[y:y + self.size, x:x + self.size]
            # D4 is resize-free, so no interpolation artefacts.
            k = random.randint(0, 3)
            if k:
                img = np.rot90(img, k)
            if random.random() < 0.5:
                img = img[:, ::-1]
        elif self.size < min(h, w):
            # Deterministic centre crop for evaluation.
            y = (h - self.size) // 2
            x = (w - self.size) // 2
            img = img[y:y + self.size, x:x + self.size]

        img = np.ascontiguousarray(img.transpose(2, 0, 1))
        tensor = torch.from_numpy(img).float().div_(255.0)

        if self.labels is None:
            return tensor
        # long, not float: CrossEntropyLoss takes class indices.
        return tensor, torch.tensor(self.labels[idx], dtype=torch.long)


def make_loader(df: pd.DataFrame, size: int, train: bool, batch_size: int) -> DataLoader:
    return DataLoader(
        ImageDataset(df, size, train),
        batch_size=batch_size,
        shuffle=train,
        drop_last=train,
        num_workers=CFG.num_workers,
        pin_memory=True,
        persistent_workers=CFG.num_workers > 0,
    )

## Model

SRM is the standard fixed second-order high-pass kernel from the forensics literature.
It is a non-trainable buffer applied per channel on the GPU, so it adds no dataloader
work.

In [ ]:
SRM_KERNEL = torch.tensor(
    [[-1.0, 2.0, -1.0],
     [2.0, -4.0, 2.0],
     [-1.0, 2.0, -1.0]]
) / 4.0

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


class ClassifierNet(nn.Module):
    """Backbone wrapper that owns preprocessing so train/infer stay consistent."""

    def __init__(self, cfg=CFG, class_prior=None):
        super().__init__()
        self.use_srm = cfg.use_srm
        self.srm_gain = cfg.srm_gain
        in_chans = 6 if self.use_srm else 3

        self.backbone = timm.create_model(
            cfg.model_name,
            pretrained=cfg.pretrained,
            in_chans=in_chans,
            num_classes=cfg.n_classes,
            drop_rate=cfg.drop_rate,
            drop_path_rate=cfg.drop_path_rate,
        )

        # A fresh head emits large logits at 640px. On Level 1 the first steps went
        # into collapsing a loss of 8.07 instead of learning, and prior-initialising
        # the head alone moved a short trial run from AUC 0.4269 to 0.8231. Zero the weight
        # and set each bias to log(prior) so it starts at the class prior, which is
        # ln 3 = 1.0986 for balanced classes.
        if class_prior is not None:
            head = self.backbone.get_classifier()
            p = np.clip(np.asarray(class_prior, dtype=np.float64), 1e-6, 1.0)
            p = p / p.sum()
            nn.init.zeros_(head.weight)
            if head.bias is not None:
                with torch.no_grad():
                    head.bias.copy_(torch.tensor(np.log(p), dtype=head.bias.dtype))

        self.register_buffer("mean", torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(IMAGENET_STD).view(1, 3, 1, 1))
        self.register_buffer("srm", SRM_KERNEL.view(1, 1, 3, 3).repeat(3, 1, 1, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: raw RGB in [0, 1], shape (B, 3, H, W). Returns logits (B, n_classes)."""
        rgb = (x - self.mean) / self.std
        if self.use_srm:
            # High-pass the raw pixels before normalisation, otherwise per-channel
            # scaling gets folded into the residual.
            residual = F.conv2d(x, self.srm, padding=1, groups=3) * self.srm_gain
            rgb = torch.cat([rgb, residual], dim=1)
        return self.backbone(rgb)

## Metrics

argmax is the decision rule and needs no fitting: the classes are exactly balanced, so
there is no prior to correct for. The confusion matrix is the view that matters here,
since it says which pair is actually costing the score.

In [ ]:
def decide(probs: np.ndarray) -> np.ndarray:
    """Argmax over the class probabilities."""
    return probs.argmax(axis=1)


def evaluate(y_true: np.ndarray, probs: np.ndarray) -> dict:
    """Metric report from (N, n_classes) probabilities."""
    pred = decide(probs)
    per_class = f1_score(y_true, pred, average=None, labels=CLASS_LABELS,
                         zero_division=0)
    out = {
        "macro_f1": float(f1_score(y_true, pred, average="macro",
                                   labels=CLASS_LABELS, zero_division=0)),
        "accuracy": float((pred == y_true).mean()),
        "errors": int((pred != y_true).sum()),
        "n": int(len(y_true)),
    }
    for name, v in zip(CLASS_NAMES, per_class):
        out[f"f1_{name}"] = float(v)
    return out


def format_metrics(m: dict) -> str:
    return (f"macroF1 {m['macro_f1']:.4f} | err {m['errors']:>4}/{m['n']} | "
            f"f1 {m['f1_real']:.4f}/{m['f1_generated']:.4f}/{m['f1_local_edit']:.4f}")


def print_confusion(y_true: np.ndarray, probs: np.ndarray) -> None:
    """Confusion matrix, the only view that shows which pair is being confused."""
    cm = confusion_matrix(y_true, decide(probs), labels=CLASS_LABELS)
    print("  confusion (rows = truth, cols = predicted)")
    print("            " + "".join(f"{n:>12}" for n in CLASS_NAMES))
    for name, row in zip(CLASS_NAMES, cm):
        print(f"  {name:>10}  " + "".join(f"{v:>12}" for v in row))


## Train one fold

In [ ]:
@torch.no_grad()
def predict(model: nn.Module, loader: DataLoader) -> np.ndarray:
    """Return softmax probabilities, shape (N, n_classes)."""
    model.eval()
    probs = []
    for batch in loader:
        x = batch[0] if isinstance(batch, (list, tuple)) else batch
        x = x.to(DEVICE, non_blocking=True)
        with autocast():
            p = torch.softmax(model(x).float(), dim=1)
        probs.append(p.cpu().numpy())
    return np.concatenate(probs)


def train_one_fold(fold_train: pd.DataFrame, fold_valid: pd.DataFrame, tag: str) -> dict:
    """Train a single model. Returns val probs, test probs and metrics."""
    print(f"\n[{tag}] train {len(fold_train)}  valid {len(fold_valid)}")

    train_loader = make_loader(fold_train, CFG.train_size, True, CFG.batch_size)
    valid_loader = make_loader(fold_valid, CFG.eval_size, False, CFG.eval_batch_size)

    counts = np.bincount(fold_train.label.values, minlength=CFG.n_classes)
    model = ClassifierNet(CFG, class_prior=counts / counts.sum()).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr,
                                  weight_decay=CFG.weight_decay)

    steps_per_epoch = max(1, len(train_loader) // CFG.grad_accum)
    total_steps = steps_per_epoch * CFG.epochs
    warmup_steps = max(2, int(total_steps * CFG.warmup_frac))
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=CFG.lr,
        total_steps=total_steps,
        pct_start=warmup_steps / total_steps,
        anneal_strategy="cos",
    )
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
    criterion = nn.CrossEntropyLoss(label_smoothing=CFG.label_smoothing)

    # Selecting on macro F1 is safe here, unlike Level 1. argmax takes no fitted
    # parameter so the checkpoint choice cannot be tied to an unfitted threshold,
    # and it matches the scored metric exactly.
    best_score, best_state = -1.0, None

    for epoch in range(CFG.epochs):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running, seen, t0 = 0.0, 0, time.time()

        for step, (x, y) in enumerate(train_loader):
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            with autocast():
                loss = criterion(model(x), y)

            scaler.scale(loss / CFG.grad_accum).backward()

            if (step + 1) % CFG.grad_accum == 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
                scale_before = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                # GradScaler skips optimizer.step() on inf/nan grads, which it does
                # on the first step at the initial scale of 65536. Stepping the LR
                # schedule for an update that never happened is wrong and triggers
                # the "step() before optimizer.step()" warning.
                stepped = scaler.get_scale() >= scale_before
                if stepped and scheduler.last_epoch < total_steps - 1:
                    scheduler.step()

            running += loss.item() * x.size(0)
            seen += x.size(0)

        val_probs = predict(model, valid_loader)
        metrics = evaluate(fold_valid.label.values, val_probs)
        print(f"  ep{epoch} {time.time() - t0:.0f}s | loss {running / max(seen, 1):.4f} "
              f"| {format_metrics(metrics)}")

        if metrics["macro_f1"] > best_score:
            best_score = metrics["macro_f1"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    # Reload the best epoch before producing the predictions actually used.
    model.load_state_dict(best_state)

    test_loader = make_loader(test_df, CFG.eval_size, False, CFG.eval_batch_size)
    val_probs = predict(model, valid_loader)
    test_probs = predict(model, test_loader)
    final_metrics = evaluate(fold_valid.label.values, val_probs)
    print(f"[{tag}] best epoch: {format_metrics(final_metrics)}")
    print_confusion(fold_valid.label.values, val_probs)

    del model, best_state
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "tag": tag,
        "val_ids": fold_valid.image_id.tolist(),
        "val_labels": fold_valid.label.values,
        "val_probs": val_probs,
        "test_probs": test_probs,
        "metrics": final_metrics,
    }


## Run

5-fold StratifiedKFold over train and val pooled, which gives more data per model
than a fixed holdout, a 5-model ensemble, and an 8835-image out-of-fold set. Level 1
showed the 978-image holdout saturates and stops being able to rank models.

In [ ]:
results = []
pool = pd.concat([train_df, val_df], ignore_index=True)
skf = StratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)

for fold, (tr_idx, va_idx) in enumerate(skf.split(pool, pool.label)):
    results.append(train_one_fold(
        pool.iloc[tr_idx].reset_index(drop=True),
        pool.iloc[va_idx].reset_index(drop=True),
        f"fold{fold}",
    ))

print(f"\ntrained {len(results)} models")


## Pooled out-of-fold results

Every prediction comes from a model that never saw the image. In cv that covers all
8835 labelled images, around 9x the resolution of the 978-image holdout.

In [ ]:
oof_labels = np.concatenate([r["val_labels"] for r in results])
oof_probs = np.concatenate([r["val_probs"] for r in results])
oof_ids = [i for r in results for i in r["val_ids"]]

# Fold test predictions are averaged; each fold is an equally valid estimator.
test_probs = np.mean([r["test_probs"] for r in results], axis=0)

oof_metrics = evaluate(oof_labels, oof_probs)
print(f"OOF n = {len(oof_labels)}")
print(f"  {format_metrics(oof_metrics)}")
print_confusion(oof_labels, oof_probs)

best_constant = max(macro_f1_of_constant_prediction(oof_labels, c)
                    for c in range(CFG.n_classes))
print(f"\nOOF macro F1              : {oof_metrics['macro_f1']:.4f}")
print(f"best constant baseline    : {best_constant:.4f}")
print(f"improvement over baseline : {oof_metrics['macro_f1'] - best_constant:+.4f}")

# Which pair is actually being confused. With class 0 near-solved on Level 1, this
# should be dominated by 1 against 2.
cm = confusion_matrix(oof_labels, decide(oof_probs), labels=CLASS_LABELS)
off_diag = [(cm[i, j], CLASS_NAMES[i], CLASS_NAMES[j])
            for i in range(CFG.n_classes) for j in range(CFG.n_classes) if i != j]
print("\nmisclassifications by pair (truth -> predicted):")
for count, truth, pred_name in sorted(off_diag, reverse=True):
    if count:
        print(f"  {count:>4}  {truth} -> {pred_name}")


## Build the submission

A malformed file is rejected without consuming a daily attempt, but a well-formed wrong
one costs one of five. Column is `classe`, and labels must be in {0, 1, 2}.

In [ ]:
def build_submission(ids: list[str], probs: np.ndarray) -> pd.DataFrame:
    return pd.DataFrame({"image_id": ids, "classe": decide(probs).astype(int)})


def validate_submission(sub: pd.DataFrame, reference: pd.DataFrame) -> None:
    """Raise on anything the grader would reject."""
    assert list(sub.columns) == ["image_id", "classe"], f"bad columns: {list(sub.columns)}"
    assert len(sub) == len(reference), f"expected {len(reference)} rows, got {len(sub)}"
    assert sub.image_id.is_unique, "duplicate image_id"
    assert set(sub.image_id) == set(reference.image_id), "image_id set does not match test/"
    assert sub.classe.dtype.kind in "iu", f"classe must be integer, got {sub.classe.dtype}"
    assert set(sub.classe.unique()) <= set(CLASS_LABELS), \
        f"classe outside {{0,1,2}}: {sub.classe.unique()}"
    assert sub.notna().all().all(), "null values present"
    print("submission validated OK")


submission = build_submission(test_df.image_id.tolist(), test_probs)
validate_submission(submission, sample_sub)

sub_path = os.path.join(CFG.out_dir, "submission.csv")
submission.to_csv(sub_path, index=False)

print(f"\nwrote {sub_path}")
dist = submission.classe.value_counts().sort_index()
print("predicted test distribution:")
for c, name in enumerate(CLASS_NAMES):
    n = int(dist.get(c, 0))
    print(f"  {name:11} {n:>4}  ({n / len(submission):.4f})")
print("Balanced thirds (~331 each) is the expected shape if test matches train.")


## Save probabilities

Level 3 mounts this notebook's predictions.npz through `kernel_sources` and uses its
argmax as the Cat-2 gate, deciding which test images get a polygon. These arrays are
an output the next level depends on.

In [ ]:
# Level 3 mounts this file and uses its argmax as the Cat-2 gate, so these arrays
# are an output the next level depends on rather than a convenience dump.
np.savez_compressed(
    os.path.join(CFG.out_dir, "predictions.npz"),
    oof_ids=np.array(oof_ids),
    oof_probs=oof_probs,
    oof_labels=oof_labels,
    test_ids=np.array(test_df.image_id.tolist()),
    test_probs=test_probs,
)
print("wrote predictions.npz")


## Notes

Class 0 is near-solved, so the confusion matrix is the first thing to read. If the
errors are mostly generated against local_edit, that pair is the whole game.

Two things worth carrying over from Level 1: 5 submissions a day of which only 2 count,
so decide from OOF rather than the public board, which has a standard error near
+/-0.015 on 993 images; and re-run an identical config once before trusting any A/B,
or the +/-2 noise floor will fool you.